In [2]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)
print("✅ Client Groq configuré !")

✅ Client Groq configuré !


In [10]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [11]:
llm("Dis bonjour en français")

'Bonjour !" est une expression courante en français pour saluer quelqu\'un. Cela signifie "bonjour" ou "bonne journée" pour le matin, et "bonsoir" ou "bonne nuit" pour l\'après-midi et le soir. \n\nVous pouvez également utiliser d\'autres expressions pour saluer en français, telles que :\n\n- Bonne journée !\n- Bonsoir !\n- Bonne nuit !\n- Salut !\n- Bon matin !\n- Enchanté(e) !'

In [17]:
llm("comment cuisiner du saumon ?")

"Le saumon est un poisson délicieux et riche en nutriments ! Voici quelques astuces pour cuisiner le saumon de manière facile et savoureuse :\n\n**Préparation du saumon**\n\nAvant de commencer à cuire le saumon, assurez-vous de le faire mariner dans un mélange de jus de citron, d'herbes fines (comme le thym ou la sarriette) et d'épices (comme du poivre noir ou du curry) pendant au moins 30 minutes. Cela aidera à rendre le saumon plus juteux et plus aromatique.\n\n**Méthodes de cuisson**\n\n1. **Grillage** : Faites griller le saumon dans un four à 200°C pendant environ 12 à 15 minutes, ou jusqu'à ce qu'il soit cuit à point. Vous pouvez également le griller sur une poêle à frire chaude pendant 3 à 5 minutes de chaque côté.\n2. **Poêlage** : Faites chauffer une poêle à feu moyen avec un peu d'huile d'olive. Ajoutez le saumon et laissez cuire pendant environ 4 à 6 minutes de chaque côté, ou jusqu'à ce qu'il soit cuit à point.\n3. **Sous-vide** : Faites cuire le saumon dans un sous-vide à 5

In [18]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [19]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [20]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [12]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

You're interested in joining a course. I'm an AI, so I don't have direct access to specific course information or enrollment systems. 

To join a course, you'll typically need to contact the course administrators, instructors, or the educational institution offering the course directly. They can provide you with information on enrollment procedures, deadlines, and any requirements you might need to meet before joining.

Can you please provide me with more details about the course you're interested in (like the course name, institution, or subject matter) and I can try to help you find contact information or direct you to a suitable resource for further assistance?


In [13]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [14]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [15]:
answer = llm(prompt)
print(answer)

Unfortunately I don't have your specific course details, however,  I can answer your question based on the context:

Yes, you can still join, but to receive a certificate, you need to submit your project while the course is still accepting submissions.


In [16]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [24]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [25]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [26]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [30]:
import requests

docs_url = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-intro/documents.json"
docs_response = requests.get(docs_url)

print("Status:", docs_response.status_code)
print("Début du contenu:", docs_response.text[:200])

Status: 404
Début du contenu: 404: Not Found


In [31]:
import requests

# Essayons une autre URL
docs_url = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-intro/documents.json"

# Vérifions d'abord si le repo existe
response = requests.get("https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/README.md")
print("Status README:", response.status_code)

Status README: 200


In [33]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
docs_response = requests.get(docs_url)
courses_raw = docs_response.json()

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"{url_prefix}{course['path']}"
    course_response = requests.get(course_url)
    course_data = course_response.json()
    documents.extend(course_data)

print(f"✅ {len(documents)} documents chargés !")

✅ 1350 documents chargés !


In [34]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)
print("✅ Index créé !")

✅ Index créé !


In [35]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [36]:
results = search("I just discovered the course. Can I join now?")
for r in results:
    print(r["question"])
    print("---")

I just discovered the course. Can I still join?
---
Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
---
Certificate: Can I follow the course in a self-paced mode and get a certificate?
---
How should I start the course and follow the weekly workflow?
---
When will the course be offered next?
---


In [37]:
def build_prompt(question, search_results):
    context = ""
    for doc in search_results:
        context += f"Question: {doc['question']}\nAnswer: {doc['answer']}\n\n"
    
    prompt = f"""
You are a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
If the answer is not found in the context, respond with "I don't know."

QUESTION: {question}

CONTEXT:
{context}
""".strip()
    
    return prompt

In [38]:
question = "I just discovered the course. Can I join now?"
results = search(question)
prompt = build_prompt(question, results)
print(prompt)

You are a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
If the answer is not found in the context, respond with "I don't know."

QUESTION: I just discovered the course. Can I join now?

CONTEXT:
Question: I just discovered the course. Can I still join?
Answer: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

Question: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
Answer: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

Question: Certificate: Can I follow the course in a self-paced mode and get a certificate?
Answer: No, you can only get a certificate if you

In [40]:
def rag(question):
    search_results = search(question)
    prompt = build_prompt(question, search_results)
    answer = llm(prompt)
    return answer

In [41]:
question = "I just discovered the course. Can I join now?"
answer = rag(question)
print(answer)

Yes, you can join the course now. As stated in the FAQ, you can start learning and submitting homework while the submission form is still open, and you can even start joining now without registering.


In [42]:
def rag(question):
    search_results = search(question)      # 🔍 Cherche
    prompt = build_prompt(question, search_results)  # 📝 Construit
    answer = llm(prompt)                   # 🤖 Génère
    return answer

In [43]:
print(rag("How do I get a certificate?"))

To get a certificate, I need to finish the course with a "live" cohort.


In [44]:
print(rag("What is the difference between self-paced and live cohort?"))

I don't know.


In [45]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.
Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")
    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [46]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [47]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.
Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")
    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [48]:
question = "I just discovered the course. Can I join now?"
search_results = search(question)
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [49]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": INSTRUCTIONS},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

In [50]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join the course. The only condition is that if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.


In [51]:
def llm(instructions, user_prompt):
    response = openai_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.choices[0].message.content

In [52]:
def rag(question):
    search_results = search(question)
    prompt = build_prompt(question, search_results)
    answer = llm(INSTRUCTIONS, prompt)
    return answer

In [53]:
print(rag("I just discovered the course. Can I join now?"))
print(rag("How do I get a certificate?"))

Yes, you can join the course now. According to the course description, "You don't need it" to register for the course to start learning, and you can submit homework without registering (as long as the registration form is open).
You can get a certificate by finishing the course with a "live" cohort.


In [54]:
print(rag("How do I get a certificate?"))

To get a certificate, what do you need to do? 

You need to finish the course with a "live" cohort.


In [55]:
from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

documents = load_faq_data()
index = build_index(documents)

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

print(assistant.rag("I just discovered the course. Can I join now?"))

You can join the course now, but if you want to receive a certificate, you need to submit your project while the course submission form is still open.


In [56]:
print(assistant.rag("How do I get a certificate?"))

You need to finish the course with a "live" cohort and pass the Capstone project to get a certificate.


In [57]:
print(assistant.rag("Can I still join the course after it started?"))

You can still join the course after it started, but keep in mind the following requirements:

- If you want to receive a certificate, you need to submit your project while the course is still accepting submissions.
- Homework is not mandatory for receiving a certificate, but completing it can help with reinforcing concepts and increasing your rank on the leaderboard.

You can also start the course whenever you want, as the videos, GitHub materials, and deadlines are available. To begin, follow these steps:

1. Start with the provided documentation: LLM Zoomcamp docs, general Zoomcamp logistics docs, and the LLM Zoomcamp GitHub repository.
2. Familiarize yourself with the course management platform for the deadlines.
3. Watch the lesson videos, work through the lesson notebooks/code, and read the homework instructions on GitHub.
4. Submit your answers through the course platform before the deadline.


In [58]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [59]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [60]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [61]:
import json

messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    tools=[search_tool],
    tool_choice="auto"
)

print(response.choices[0].message)

BadRequestError: Error code: 400 - {'error': {'message': 'code=400, message=tools[0].function.name is required, type=invalid_request_error', 'type': 'invalid_request_error'}}

In [62]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"]
        }
    }
}

In [63]:
import json

messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    tools=[search_tool],
    tool_choice="auto"
)

print(response.choices[0].message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='qd9f0b6fn', function=Function(arguments='{"query":"course enrollment instructions"}', name='search'), type='function')])


In [64]:
tool_call = response.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

messages.append(response.choices[0].message)
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": result_json
})

response2 = openai_client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    tools=[search_tool]
)

print(response2.choices[0].message.content)

You can still join the course, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.


In [65]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [66]:
import json

def make_call(tool_call):
    args = json.loads(tool_call.function.arguments)
    
    if tool_call.function.name == "search":
        result = search(**args)
    
    result_json = json.dumps(result, indent=2)
    
    return {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result_json
    }

In [70]:
def agent_loop(instructions, question):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            tools=[search_tool],
            tool_choice="auto"
        )

        msg = response.choices[0].message
        messages.append(msg)

        if msg.tool_calls:
            for tool_call in msg.tool_calls:
                print("function_call:", tool_call.function.name, tool_call.function.arguments)
                call_output = make_call(tool_call)
                messages.append(call_output)
                has_function_calls = True
        else:
            print("ASSISTANT:")
            print(msg.content)
            last_answer = msg.content

        it += 1

        # sécurité : max 10 itérations
        if not has_function_calls or it > 10:
            break

    return last_answer

In [71]:
agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"joining the course"}
iteration #2...
ASSISTANT:
You can join the course. However, if you want to receive a certificate, you need to submit your project while the course is still accepting submissions. 

Would you like to explore the logistics of the course, its topics, or other areas?


'You can join the course. However, if you want to receive a certificate, you need to submit your project while the course is still accepting submissions. \n\nWould you like to explore the logistics of the course, its topics, or other areas?'

In [73]:
agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"joining the course late"}
iteration #2...
function_call: search {"query":"certificate self-paced mode"}
iteration #3...
ASSISTANT:
You are welcome to join the course, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions. Please note that you can only get a certificate if you finish the course with a "live" cohort. The self-paced mode is not eligible for certification, as it lacks the peer-review component.


'You are welcome to join the course, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions. Please note that you can only get a certificate if you finish the course with a "live" cohort. The self-paced mode is not eligible for certification, as it lacks the peer-review component.'

In [74]:
agent_loop(instructions, "How do I get a certificate?")

iteration #1...
function_call: search {"query":"course certificate"}
function_call: search {"query":"how to get certificate"}
function_call: search {"query":"obtaining certificate"}
function_call: search {"query":"certificate details"}
iteration #2...
ASSISTANT:
To get a certificate, you need to complete the course with a "live" cohort. This means you cannot follow the course in a self-paced mode and get a certificate. Additionally, you must pass the Capstone project to get the certificate. Homework is not mandatory, but it is recommended for reinforcing concepts and the points awarded count towards your rank on the leaderboard.

To submit your project and receive a certificate, you must do so while the course is still accepting submissions. If you miss the first homework, you can still get a certificate by passing the Capstone project.

Your capstone project will be evaluated by three randomly assigned students who have also submitted the project. You will also be responsible for grad

'To get a certificate, you need to complete the course with a "live" cohort. This means you cannot follow the course in a self-paced mode and get a certificate. Additionally, you must pass the Capstone project to get the certificate. Homework is not mandatory, but it is recommended for reinforcing concepts and the points awarded count towards your rank on the leaderboard.\n\nTo submit your project and receive a certificate, you must do so while the course is still accepting submissions. If you miss the first homework, you can still get a certificate by passing the Capstone project.\n\nYour capstone project will be evaluated by three randomly assigned students who have also submitted the project. You will also be responsible for grading the projects from three fellow students yourself. The final grade you receive will be the median score of the grades from the peer reviewers.\n\nIf you want to receive a certificate, you must submit your project while the course is still accepting submis

In [72]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
function_call: search {"query":"gambit"}
function_call: search {"query":"queen gambit course"}
function_call: search {"query":"gambit course logistics"}
iteration #2...
ASSISTANT:
I think your question is off-topic.

Is there any other area you would like to explore?


'I think your question is off-topic.\n\nIs there any other area you would like to explore?'